# Stage 10/11 (repurposed) — General F&O trend-following ranking

`score_for_trend_following()` in `strategy_scoring.py` — ranks the
full F&O stock+index 1D universe (see `trend_universe.py` and README
Stage 10 for the repurposing history) for what it's actually for: buy the
share on a fresh UP flip, exit on the opposite DOWN flip, pure cash equity.
Different from the other four scorers in this project — favors a large
favorable move on the UP flips specifically (buying the underlying has no
structural cap the way credit spreads/short IC's containment objective or
debit spread/option buying's ATR-bounded structure do). No stop-loss:
`src/stop_loss_test.py` found no candidate stop rule beats holding to the
natural DOWN-flip exit, so this scores the actual strategy, not a
hypothetical stopped-out version of it.

`security_type` ("Stock"/"Index") is retained purely as a display/filter
column — the ranking is one table across the whole universe, not split by
type (that split existed when the universe included ETFs, which had
structurally smaller typical moves; there are no ETFs in the universe now).

Requires notebook 10 to have been run with real deep history first — if
`up_closed_count` looks thin across the board, that's the symptom of the
now-fixed regression documented in README Stage 10/11 — re-run notebook 10.

In [1]:
import sys
sys.path.append("..")

from src.strategy_scoring import score_for_trend_following, save_ranking

ranked, watchlist = score_for_trend_following()
print(f"Ranked: {len(ranked)}, Watchlist: {len(watchlist)}")
ranked["security_type"].value_counts()

Ranked: 202, Watchlist: 16


security_type
Stock    196
Index      6
Name: count, dtype: int64

In [ ]:
# Notebook 13 (live screener) reads these two files to merge trend_score
# into its status table. This save step was accidentally dropped when the
# pledged-1D notebooks were repurposed (commit cb9dbea) - the old notebook
# saved RANKING_/WATCHLIST_pledged_trend_following.xlsx right here.
save_ranking(ranked, "RANKING_trend_following.xlsx")
save_ranking(watchlist, "WATCHLIST_trend_following.xlsx")

## Ranked universe

In [2]:
ranked.reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,BAJFINANCE,Stock,91.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.33pp),,39,69.2,12.12,7.79,3.72
1,BAJAJHLDNG,Stock,88.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.44pp),,31,71.0,8.01,4.57,4.31
2,HDFCBANK,Stock,88.3,,,38,63.2,4.19,3.28,3.25
3,TITAN,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.30pp),,36,63.9,9.56,5.26,3.73
4,AXISBANK,Stock,87.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.33pp),,35,65.7,6.74,3.41,3.72
...,...,...,...,...,...,...,...,...,...,...
197,KPITTECH,Stock,6.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 17.90pp),⚠ HIGH GAP RISK (>10.0%),24,37.5,15.64,-2.26,7.12
198,JSWENERGY,Stock,6.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 18.64pp),,38,36.8,16.68,-1.96,7.24
199,INOXWIND,Stock,4.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 11.84pp),,36,41.7,8.06,-3.78,11.66
200,CDSL,Stock,4.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 15.19pp),,29,37.9,11.62,-3.57,7.64


In [3]:
ranked[ranked["security_type"] == "ETF"].reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up


In [4]:
# Instruments below the 15 UP-closed-flip floor - not blended into the
# ranking, but not dropped either. Re-check as more calendar time passes.
print(f"On watchlist: {len(watchlist)}")
watchlist.sort_values("avg_pct_move_up", ascending=False).reset_index(drop=True)

On watchlist: 16


,symbol,security_type,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,ATHERENERG,Stock,3,66.7,72.94,107.90,6.53
1,ANANDRATHI,Stock,13,61.5,34.17,0.51,4.40
2,KAYNES,Stock,11,81.8,30.18,10.86,8.69
3,ETERNAL,Stock,11,54.5,24.99,5.37,7.55
4,HYUNDAI,Stock,4,50.0,11.51,4.08,7.82
5,KFINTECH,Stock,12,58.3,10.29,1.18,5.01
6,DELHIVERY,Stock,11,63.6,9.22,2.33,5.10
7,IREDA,Stock,10,40.0,8.44,-1.10,6.26
8,NYKAA,Stock,13,46.2,5.50,-1.35,4.51
9,ENRIN,Stock,6,50.0,4.40,3.28,6.75
